In [4]:
import os
os.environ['SPARK_VERSION'] = '3.3'
os.environ["JAVA_HOME"] = '/usr/lib/jvm/java-8-openjdk-amd64/'

In [5]:
from pyspark.sql import SparkSession
import pydeequ

spark = SparkSession.builder \
    .appName("Loan Data ETL Pipeline") \
    .master("local[*]") \
    .config(
            "spark.jars.packages", "com.amazon.deequ:deequ:2.0.11-spark-3.3"
        )\
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.memory.fraction", "0.2")
spark.conf.set("spark.sql.execution.arrow.pyspark.memory.max", "2g") 

In [6]:
from pyspark.sql.functions import col, isnan, when, count, lit, split, to_date, hour, minute, second
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.window import Window
from pyspark.sql import functions as F
# from pyspark.sql.functions import mode
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import os
from datetime import datetime


In [7]:
schema = StructType([
    StructField("Loan_id", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Married", StringType(), True),
    StructField("Dependents", IntegerType(), True),
    StructField("Education", StringType(), True),
    StructField("Self_Employed", StringType(), True),
    StructField("ApplicantIncome", IntegerType(), True),
    StructField("CoapplicantIncome", IntegerType(), True),
    StructField("LoanAmount", IntegerType(), True),
    StructField("Loan_Amount_Term", IntegerType(), True),
    StructField("Credit_History", IntegerType(), True),
    StructField("Property_Area", StringType(), True),
    StructField("Loan_Status", StringType(), True),
])

In [8]:
try:
    input_path = "hdfs://localhost:9000/user/hive/warehouse/loan.csv"
    # Read the CSV file with the specified schema
    raw_df = spark.read.csv(input_path, header=True, schema=schema)

    print("CSV file read successfully.")
    print(f"Sample data")
    raw_df.show(5, truncate=False)

    # print("Initial data statistics:")
    # raw_df.describe().show()
    
    null_counts = raw_df.select([count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in raw_df.columns])
    null_counts.show(truncate=False)
    # null_counts.show()
except Exception as e:
    print(f"Error reading CSV file: {e}")

CSV file read successfully.
Sample data
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|Loan_id |Gender|Married|Dependents|Education   |Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|Male  |No     |0         |Graduate    |No           |5849           |0                |null      |360             |1             |Urban        |Y          |
|LP001003|Male  |Yes    |1         |Graduate    |No           |4583           |1508             |128       |360             |1             |Rural        |N          |
|LP001005|Male  |Yes    |0         |Graduate    |Yes          |3000           |0                |66        |360             |

In [9]:
raw_df.select(F.min("ApplicantIncome").alias("min application income")).show()

+----------------------+
|min application income|
+----------------------+
|                   150|
+----------------------+



In [10]:
import pydeequ
from pydeequ.analyzers import AnalysisRunner, Size, Completeness, ApproxCountDistinct, Mean, AnalyzerContext, Compliance

In [11]:
analysisResult = AnalysisRunner(spark) \
                    .onData(raw_df) \
                    .addAnalyzer(Size()) \
                    .addAnalyzer(Completeness("Married")) \
                    .addAnalyzer(ApproxCountDistinct("Loan_id")) \
                    .addAnalyzer(Mean("LoanAmount")) \
                    .addAnalyzer(Compliance("Min ApplicantIncome", "ApplicantIncome >= 1000")) \
                    .run()
                    # .addAnalyzer(Correlation("total_votes", "star_rating")) \
                    # .addAnalyzer(Correlation("total_votes", "helpful_votes")) \
                    
analysisResult_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysisResult)
analysisResult_df.show()

+-------+-------------------+-------------------+------------------+
| entity|           instance|               name|             value|
+-------+-------------------+-------------------+------------------+
| Column|         LoanAmount|               Mean|146.41216216216216|
|Dataset|                  *|               Size|             614.0|
| Column|            Married|       Completeness| 0.995114006514658|
| Column|Min ApplicantIncome|         Compliance|  0.99185667752443|
| Column|            Loan_id|ApproxCountDistinct|             607.0|
+-------+-------------------+-------------------+------------------+



/home/rohitkarki/.local/lib/python3.10/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [13]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult

In [ ]:
# metrics_file = FileSystemMetricsRepository.helper_metrics_file(spark, 'metrics.json')
# repository = FileSystemMetricsRepository(spark, metrics_file)
# key_tags = {'tag': 'pydeequ hello world'}
# resultKey = ResultKey(spark, ResultKey.current_milli_time(), key_tags)


check = Check(spark, CheckLevel.Error, "Loan Dataset Review Check")

checkResult = VerificationSuite(spark)\
    .onData(raw_df)\
    .addCheck(
        check.isComplete("Loan_id") \
        .isUnique("Loan_id") \
        .hasCompleteness("Married", assertion=lambda x: x==1) \
        .isPositive("ApplicantIncome") \
        .isComplete("Gender", assertion ) \
        .isContainedIn("Gender", ["Male", "Female"], lambda x: x==1)) \
        .run()

        # .useRepository(repository) \
        # .saveOrAppendResult(resultKey) \

print(f"Verification Run Status: {checkResult.status}")

# Checking the results of the verification
checkResult_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)

checkResult_df.show(truncate=False)
raw_df.fillna("Male")


Verification Run Status: Error
+-------------------------+-----------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+-------------------------------------------------------------------+
|check                    |check_level|check_status|constraint                                                                                                                                              |constraint_status|constraint_message                                                 |
+-------------------------+-----------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+-------------------------------------------------------------------+
|Loan Dataset Review Check|Error      |Error       |CompletenessConstraint(Completeness(Loan_

DataFrame[Loan_id: string, Gender: string, Married: string, Dependents: int, Education: string, Self_Employed: string, ApplicantIncome: int, CoapplicantIncome: int, LoanAmount: int, Loan_Amount_Term: int, Credit_History: int, Property_Area: string, Loan_Status: string]

In [4]:
from pydeequ.suggestions import *

In [11]:
suggestionResult = ConstraintSuggestionRunner(spark) \
            .onData(raw_df) \
            .addConstraintRule(DEFAULT()) \
            .run()

# Constraint Suggestions in JSON format
print(suggestionResult)

Py4JError: An error occurred while calling None.com.amazon.deequ.suggestions.rules.FractionalCategoricalRangeRule. Trace:
py4j.Py4JException: Constructor com.amazon.deequ.suggestions.rules.FractionalCategoricalRangeRule([class java.lang.Double, class com.amazon.deequ.suggestions.rules.FractionalCategoricalRangeRule$$$Lambda$3216/1312856326]) does not exist
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:179)
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:196)
	at py4j.Gateway.invoke(Gateway.java:237)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.lang.Thread.run(Thread.java:750)

